<a href="https://colab.research.google.com/github/awaiskhan005/DATA-SCIENCE-AND-AI-/blob/main/Boston_Housing_EDA_%26_Regression_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
arslanali4343_real_estate_dataset_path = kagglehub.dataset_download('arslanali4343/real-estate-dataset')

print('Data source import complete.')


# **Boston Housing Price**

# import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('ggplot')

from sklearn.preprocessing import scale, StandardScaler,OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, VotingRegressor,BaggingRegressor
from xgboost import XGBRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC
from sklearn.model_selection import KFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_selection import SelectFromModel
from imblearn.pipeline import Pipeline as imbpipeline

import warnings
warnings.filterwarnings('ignore')

# Read Data

In [ ]:
df = pd.read_csv("/kaggle/input/real-estate-dataset/data.csv")
df.head()

### Feature Description

**key features in the Boston Housing Dataset:**
- CRIM: Per capita crime rate by town
- ZN: Proportion of residential land zoned for large lots
- INDUS: Proportion of non-retail business acres per town
- CHAS: Charles River dummy variable (1 if tract bounds river; 0 otherwise)
- NOX: Nitric oxides concentration (parts per 10 million)
- RM: Average number of rooms per dwelling
- AGE: Proportion of owner-occupied units built prior to 1940
- DIS: Weighted distances to five Boston employment centers
- RAD: Index of accessibility to radial highways
- TAX: Full-value property tax rate per $10,000
- PTRATIO: Pupil-teacher ratio by town
- B: 1000(Bk-0.63)^2 where Bk is the proportion of Black residents by town

- LSTAT: Percentage of lower status of the population

- MEDV: Median value of owner-occupied homes in $1000s (the target variable)

# EDA

In [ ]:
df.info()

In [ ]:
df.describe()

### Check Missing Values

In [ ]:
df.isna().sum()

- insight : there is 5 record missing in **RM**, so i will fill it by the median value

In [ ]:
df['RM'] = df['RM'].fillna(df['RM'].median())

In [ ]:
df.isna()

In [ ]:
df.isna().any().sum()

### Check duplicated Rows

In [ ]:
df.duplicated().sum()

- insight : there is no duplicated Rows

# Univariate Analysis

### Exploring Target Feature

In [ ]:
sns.scatterplot(data=df['MEDV'])

In [ ]:
sns.histplot(data=df['MEDV'],kde=True,palette="YlGnBu")

- insight: MEDV is more distributed between 10.000 and 30.000  $

### Feature Distrubtion

In [ ]:
plt.figure(figsize=(20,18))
sns.set_style("white")
sns.set_palette("bright")
plt.subplots_adjust(hspace=0.5)
i = 1;
for name in df.columns:
    plt.subplot(5,3,i)
    sns.histplot(data=df, x=name,palette="pastel")
    i = i + 1

In [ ]:
plt.figure(figsize=(20,18))
sns.set_style("white")
sns.set_palette("bright")
plt.subplots_adjust(hspace=0.5)
i = 1;
for name in df.columns:
    plt.subplot(5,3,i)
    sns.boxplot(data=df, x=name,palette="pastel")
    i = i + 1

**Summary Inights** :
- The Average of Per capita crime rate by town is Low
- The Average of Proportion of residential land zoned for large lots is Low
- The Average of Proportion of Non-Retail Business Acres is 11.15 %
- The percentage of tract bounds the Charles River is high

# Bi-variate Analysis

### Each Features vs Target Feature

In [ ]:
plt.figure(figsize=(20,18))
sns.set_style("white")
sns.set_palette("bright")
plt.subplots_adjust(hspace=0.5)
i = 1;
for name in df.columns:
    plt.subplot(5,3,i)
    sns.scatterplot(data=df, x='MEDV',y=name,palette="pastel")
    i = i + 1

# Multivariate Analysis

### correlation between each features

In [ ]:
plt.figure(figsize=(12,12))
sns.heatmap(df.corr(),annot=True,cmap='Oranges',cbar=False)

# Data Preprocessing

### Split data to train and validation

In [ ]:
y = df['MEDV']
x = df.drop('MEDV',axis=1)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

### Scaling by StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
X_train = pd.DataFrame(x_train_scaled, columns=x_train.columns)

x_test_scaled = scaler.transform(x_test)
X_test = pd.DataFrame(x_test_scaled, columns=x_test.columns)

### handling outliers by using winsorize


In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 5))

X_train.boxplot(ax=axes[0])
axes[0].set_title('x_train Distribution')
X_test.boxplot(ax=axes[1])
axes[1].set_title('x_test Distribution')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats.mstats import winsorize

for column_name in x_train.columns:

  X_train[column_name] = winsorize(X_train[column_name], limits=[0.05, 0.05])
  X_train[column_name] = np.tanh(X_train[column_name])
  X_test[column_name] = winsorize(X_test[column_name], limits=[0.05, 0.05])
  X_test[column_name] = np.tanh(X_test[column_name])

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 5))

X_train.boxplot(ax=axes[0])
axes[0].set_title('x_train Distribution')
X_test.boxplot(ax=axes[1])
axes[1].set_title('x_test Distribution')

plt.tight_layout()
plt.show()

# **comparing different models**

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

def compare_regression_models(models, X_train, y_train, X_test, y_test):
    """
    Compare different regression models.

    Parameters:
    - models: A dictionary containing model names as keys and model objects as values.
    - X_train, y_train: Training data.
    - X_test, y_test: Test data.

    Returns:
    - Dictionary containing model names as keys and mean squared error on the test set as values.
    """
    results = {}

    for model_name, model in models.items():
        # Train the model on the training set
        model.fit(X_train, y_train)

        # Make predictions on the test set
        y_pred = model.predict(X_test)

        # Evaluate the model using mean squared error
        mse = mean_squared_error(y_test, y_pred)

        # Store the results
        results[model_name] = mse

        # Print the results
        print(f"{model_name} - Mean Squared Error: {mse}")

    return results

models = {
    'Linear Regression': LinearRegression(),
    'RandomForest Regression': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBRegressor': XGBRegressor(n_estimators=100, random_state=42),
    'SVR': SVR(kernel='linear', C=1.0),
    'Gradient Boosting Regression': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

# Compare models
results = compare_regression_models(models, X_train, y_train, X_test, y_test)

# Print the model with the lowest mean squared error
best_model = min(results, key=results.get)
print(f"\nBest Model: {best_model} - Mean Squared Error: {results[best_model]}")


### Hyperparameter tuning for RandomForestRegressor by Optuna

In [ ]:
import optuna
from sklearn.metrics import mean_squared_error

def objective(trial):
    # Define the hyperparameter search space
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    random_state = trial.suggest_int('random_state', 1, 200)
    max_depth = trial.suggest_int('max_depth', 5, 30)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
    max_features = trial.suggest_categorical('max_features', ['auto', 'sqrt', 'log2'])

    # Create and train the Random Forest model with the suggested hyperparameters
    rf_model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=random_state
    )
    rf_model.fit(X_train, y_train)

    # Predict on the test set
    y_pred = rf_model.predict(X_test)

    # Evaluate accuracy
    accuracy = mean_squared_error(y_test, y_pred)

    return accuracy

# Set up the Optuna study
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=1)

# Get the best hyperparameters
best_params = study.best_params
print("Best Hyperparameters:", best_params)

# Train the model with the best hyperparameters
best_rf_model = RandomForestRegressor(
    n_estimators=best_params['n_estimators'],
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    min_samples_leaf=best_params['min_samples_leaf'],
    max_features=best_params['max_features'],
    random_state=best_params['random_state']
)
best_rf_model.fit(X_train, y_train)

# Predict on the test set
y_pred = best_rf_model.predict(X_test)

# Evaluate accuracy
accuracy = mean_squared_error(y_test, y_pred)
print("Accuracy on Test Set:", accuracy)

In [ ]:
rf_model = RandomForestRegressor(
        n_estimators=53,
        max_depth=27,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features='log2',
        random_state=74
    )
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mse

### Hyperparameter tuning for XGBRegressor by Optuna

In [ ]:
#!pip install --quiet optuna
import optuna
from xgboost import XGBRegressor

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 0, 20),
        'learning_rate': trial.suggest_loguniform('learning_rate', 0.01, 0.1),
        'subsample': trial.suggest_float('subsample', 0.1, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.1, 1.0),
        'random_state': trial.suggest_int('random_state', 0, 200),
    }

    model = XGBRegressor(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = mean_squared_error(y_test, y_pred)
    return accuracy

# Perform hyperparameter optimization with Optuna
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=1)

# Print the best hyperparameters and corresponding accuracy
best_params = study.best_params
best_accuracy = study.best_value
print("Best Hyperparameters:", best_params)
print("Best Accuracy:", best_accuracy)

# Use the best model for predictions
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

# Evaluate the performance on the test set
accuracy = mean_squared_error(y_test, y_pred)
print("Test Set Accuracy:", accuracy)

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=121 ,
    max_depth=5 ,
    learning_rate=0.08893388011125987 ,
    subsample= 0.4464397916454239,
    colsample_bytree=0.7193975102657935,
    random_state= 138
)

xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mse

### Hyperparameter tuning for ExtraTreesRegressor by Optuna

In [ ]:
# !pip install --quiet optuna
# import optuna

def objective(trial):
    # Define hyperparameters to be optimized
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features' : trial.suggest_categorical('max_features', ['auto', 'sqrt', 'log2']),
        'random_state': trial.suggest_int('random_state', 0, 200),

    }

    # Create ExtraTreesRegressor with the suggested hyperparameters
    model = ExtraTreesRegressor(**params)

    # Fit the model on the training set
    model.fit(X_train, y_train)

    # Predict on the testing set
    y_pred = model.predict(X_test)

    # Calculate mean squared error on the testing set
    mse = mean_squared_error(y_test, y_pred)

    # Return the mean squared error as the objective value
    return mse

# Set up Optuna study
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=1)

# Print the best hyperparameters and their corresponding score
best_params = study.best_params
best_score = study.best_value
print(f"Best Hyperparameters: {best_params}")
print(f"Best Mean Squared Error: {best_score}")

# Retrieve the best model using the best hyperparameters
best_model = ExtraTreesRegressor(**best_params)
best_model.fit(X_train, y_train)

In [ ]:
EX_model = ExtraTreesRegressor(
        n_estimators=103,
        max_depth = 17,
        min_samples_leaf=1,
        min_samples_split=2 ,
        max_features='sqrt',
        random_state=104
    )
EX_model.fit(X_train, y_train)
y_pred = EX_model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mse, np.sqrt(mse)

# Ensemple Model

In [ ]:
# Create a Voting Regressor
voting_model = VotingRegressor(estimators=[('rf', rf_model), ('xgb', xgb_model), ('EX', EX_model),])

# Train the models
rf_model.fit(X_train, y_train.ravel())
EX_model.fit(X_train, y_train.ravel())
xgb_model.fit(X_train, y_train.ravel())
voting_model.fit(X_train, y_train.ravel())

# Make predictions on the test set
y_pred_rf = rf_model.predict(X_test)
y_pred_EX = EX_model.predict(X_test)
y_pred_xgb = xgb_model.predict(X_test)
y_pred_voting = voting_model.predict(X_test)

# Evaluate the models using mean squared error
mse_rf = mean_squared_error(y_test, y_pred_rf)
mse_EX = mean_squared_error(y_test, y_pred_EX)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
mse_voting = mean_squared_error(y_test, y_pred_voting)

print(f"Random Forest MSE: {mse_rf}")
print(f"Extra Trees MSE: {mse_EX}")
print(f"XGBoost MSE: {mse_xgb}")
print(f"Voting Regressor MSE: {mse_voting}")

# Save Model

In [ ]:
import joblib

# Save the model
joblib.dump(voting_model, 'voting_model.joblib')